# Room Redesign — SDXL + ControlNet

**Input:** 1 ảnh phòng (trống hoặc đã có đồ) + chọn *room type* + *style*

**Output:** ảnh phòng render lại đầy đủ nội thất theo style, giữ nguyên kiến trúc (tường, cửa sổ, cửa, tỉ lệ phòng)

**Kỹ thuật:**
- SDXL + checkpoint chuyên nội thất (RealVisXL / Juggernaut XL) — chất lượng ảnh cao hơn nhiều SD1.5
- **ControlNet-Canny** giữ đường kiến trúc → chừa không gian trống cho model đặt đồ có khối thật
- **ControlNet-Depth** (chỉ bật khi phòng đã có đồ) giữ bố cục đồ đạc cũ
- Render giữ đúng tỉ lệ ảnh gốc (~1 megapixel), không bóp về ảnh vuông

> **Bắt buộc:** Runtime > Change runtime type > **GPU** (T4 là đủ).
> Thời gian: ~60-120 giây/ảnh trên T4 free với 1 ControlNet.


## 1. Cài đặt thư viện

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate safetensors peft
!pip install -q controlnet_aux opencv-python-headless


## 2. Import & load model

Lần đầu chạy sẽ tải ~8-10 GB (SDXL + 2 ControlNet), mất 3-6 phút.

- `BASE_MODEL`: checkpoint SDXL. RealVisXL V5 và Juggernaut XL v9 đều mạnh về nội thất/kiến trúc; `stabilityai/stable-diffusion-xl-base-1.0` là bản gốc (an toàn nhất nhưng ảnh "AI" hơn).
- `use_small_controlnet`: bản ControlNet rút gọn của diffusers, nhẹ VRAM hơn hẳn — nên bật trên T4 free. Tắt nếu chạy trên A100/L4 để có độ bám sát cao hơn.


In [ ]:
#@title Load SDXL + ControlNet { display-mode: "form" }
BASE_MODEL = "SG161222/RealVisXL_V5.0"  #@param ["SG161222/RealVisXL_V5.0", "RunDiffusion/Juggernaut-XL-v9", "stabilityai/stable-diffusion-xl-base-1.0"]
use_small_controlnet = True  #@param {type:"boolean"}
load_depth_controlnet = True  #@param {type:"boolean"}

import gc
import numpy as np
import torch
from PIL import Image
from diffusers import (
    StableDiffusionXLControlNetPipeline,
    ControlNetModel,
    AutoencoderKL,
    UniPCMultistepScheduler,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
if device == "cpu":
    print("CẢNH BÁO: không thấy GPU. Runtime > Change runtime type > GPU, rồi chạy lại.")

if use_small_controlnet:
    CANNY_ID = "diffusers/controlnet-canny-sdxl-1.0-small"
    DEPTH_ID = "diffusers/controlnet-depth-sdxl-1.0-small"
else:
    CANNY_ID = "diffusers/controlnet-canny-sdxl-1.0"
    DEPTH_ID = "diffusers/controlnet-depth-sdxl-1.0"

# ---- depth estimator (chỉ cần khi xử lý phòng đã có đồ) ----
depth_estimator = None
if load_depth_controlnet:
    from controlnet_aux import MidasDetector
    print("Đang tải depth estimator (MiDaS)...")
    depth_estimator = MidasDetector.from_pretrained("lllyasviel/Annotators")

# ---- ControlNet ----
print(f"Đang tải ControlNet-Canny: {CANNY_ID}")
controlnet_canny = ControlNetModel.from_pretrained(CANNY_ID, torch_dtype=dtype, variant=None)

controlnet_depth = None
if load_depth_controlnet:
    print(f"Đang tải ControlNet-Depth: {DEPTH_ID}")
    controlnet_depth = ControlNetModel.from_pretrained(DEPTH_ID, torch_dtype=dtype, variant=None)

# ---- VAE: bản fp16-fix, tránh ảnh ra đen/NaN khi chạy fp16 trên T4 ----
print("Đang tải VAE fp16-fix...")
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=dtype)

# ---- Pipeline: nạp danh sách 2 controlnet, lúc gọi sẽ chọn dùng 1 hay 2 ----
controlnets = [controlnet_canny] + ([controlnet_depth] if controlnet_depth is not None else [])

print(f"Đang tải SDXL: {BASE_MODEL}")
def _load_pipe(variant):
    return StableDiffusionXLControlNetPipeline.from_pretrained(
        BASE_MODEL,
        controlnet=controlnets,
        vae=vae,
        torch_dtype=dtype,
        use_safetensors=True,
        variant=variant,
    )

# Nhieu checkpoint community khong up ban fp16 rieng -> thu fp16 truoc, khong co thi lay ban full
try:
    pipe = _load_pipe("fp16" if dtype == torch.float16 else None)
except Exception as e:
    print(f"  Khong co variant fp16 ({type(e).__name__}), tai ban day du...")
    pipe = _load_pipe(None)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

# Tiết kiệm VRAM trên T4 16GB: offload từng module sang CPU khi không dùng
pipe.enable_model_cpu_offload()

# VAE slicing/tiling: diffusers moi bo 2 method nay o cap pipeline, chuyen xuong pipe.vae
for _name in ("enable_slicing", "enable_tiling"):
    if hasattr(pipe.vae, _name):
        getattr(pipe.vae, _name)()
    elif hasattr(pipe, f"enable_vae_{_name.split('_')[1]}"):
        getattr(pipe, f"enable_vae_{_name.split('_')[1]}")()

gc.collect()
torch.cuda.empty_cache() if device == "cuda" else None

CONTROL_ORDER = ["canny"] + (["depth"] if controlnet_depth is not None else [])
print(f"\nModel sẵn sàng. ControlNet đang nạp: {CONTROL_ORDER}")


## 3. Room type & Style
- `ROOM_TYPES`: loại phòng -> mô tả tiếng Anh + nội thất đặc trưng (giúp model đặt đúng đồ vào đúng phòng).
- `STYLE_PROMPTS`: phong cách -> mô tả vật liệu / màu / ánh sáng.
- Prompt cuối = style + room + nội thất + quality tag. Thêm loại phòng / style mới thì thêm 1 dòng vào dict, và thêm vào danh sách `#@param` ở bước 5.


In [ ]:
# ---------- 3.1 Loại phòng ----------
# key = nhãn hiển thị, value = (mô tả phòng tiếng Anh, nội thất đặc trưng)
ROOM_TYPES = {
    "Phòng khách":       ("living room",            "large sofa, coffee table, armchair, area rug, floor lamp, framed wall art, curtains"),
    "Phòng ngủ":         ("bedroom",                "double bed with headboard, bedding, nightstands, bedside lamps, wardrobe, area rug, curtains"),
    "Phòng ăn":          ("dining room",            "dining table, dining chairs, pendant light above table, sideboard, centerpiece"),
    "Phòng tắm":         ("bathroom",               "bathtub, vanity with mirror, walk-in shower, wall tiles, towels, small plant"),
    "Bếp":               ("kitchen",                "kitchen cabinets, stone countertop, kitchen island, bar stools, tile backsplash, range hood"),
    "Phòng chơi game":   ("gaming room",            "gaming desk, gaming chair, dual monitors, led strip lighting, wall shelves, bean bag"),
    "Nhà hàng":          ("restaurant interior",    "multiple dining tables and chairs, bar counter, decorative pendant lights, banquette seating"),
    "Văn phòng tại gia": ("home office",            "desk, ergonomic chair, bookshelf, task lamp, potted plants, framed art"),
    "Quán cà phê":       ("coffee shop interior",   "cafe counter, espresso machine, small round tables, bentwood chairs, menu board, pendant lights"),
    "Văn phòng":         ("corporate office",       "workstations, office desks, task chairs, meeting table, acoustic panels, carpet tiles"),
}

# ---------- 3.2 Phong cách ----------
STYLE_PROMPTS = {
    "Peaceful":         "peaceful serene style, warm taupe walls, charcoal grey sofa, thick textured wool rug, "
                        "large abstract triptych art, sheer curtains, soft diffused daylight, muted olive accents, dark walnut wood",
    "Farmhouse":        "modern farmhouse style, shiplap walls, reclaimed wood beams, warm neutral tones, vintage accents, cozy",
    "Clean Bright":     "clean bright style, crisp white walls, bright even daylight, uncluttered, fresh and airy, light wood",
    "Contemporary":     "contemporary style, sleek furniture, mixed textures, muted palette with bold accent, designer lighting",
    "Fresh Airy":       "fresh airy style, sheer curtains, pale palette, lots of daylight, indoor plants, breezy open feel",
    "Eclectic":         "eclectic style, bold color mix, patterned textiles, gallery wall, vintage and modern pieces mixed, playful",
    "Elegant":          "elegant style, refined furniture, soft neutral palette, silk and velvet textiles, subtle gold details, symmetry",
    "Minimalist":       "minimalist style, clean lines, neutral color palette, very few objects, hidden storage, calm empty surfaces",
    "Minimal Tranquil": "minimal tranquil style, warm off-white tones, soft diffused light, natural linen, zen calm, almost no decor",
    "Cartoon":          "cartoon illustration style, flat bold colors, thick outlines, playful stylized furniture, cel shaded, 2d render",
    "Scandinavian":     "scandinavian style, light wood floor, white walls, cozy textiles, hygge atmosphere, soft natural light",
    "Simple Calm":      "simple calm style, soft beige and grey palette, low profile furniture, uncluttered, gentle lighting",
    "Bright Soothing":  "bright soothing style, pastel palette, rounded soft furniture, warm sunlight, comfortable and relaxing",
    "Cyberpunk":        "cyberpunk style, neon pink and cyan lighting, dark surfaces, holographic panels, futuristic tech decor, moody",
    "Rustic":           "rustic style, exposed wooden beams, stone wall, timber furniture, woven textiles, fireplace, earthy tones",
    "Compact Calm":     "compact calm style, small space smart layout, multifunctional furniture, light palette, tidy and efficient",
    "Classic Graceful": "classic graceful style, wall moulding, ornate but restrained furniture, cream and soft blue palette, chandelier",
    "Traditional":      "traditional style, dark wood furniture, patterned rug, symmetrical layout, warm lamps, framed artwork",
    "Industrial":       "industrial style, exposed brick wall, black metal fixtures, concrete floor, leather furniture, edison bulbs",
    "Mid-Century":      "mid century modern style, teak furniture, tapered legs, mustard and olive accents, geometric patterns, 1960s design",
    "Japandi":          "japandi style, japanese and scandinavian fusion, natural wood, minimal decor, tatami accents, soft neutral tones",
    "Luxury Classic":   "luxury classic interior, marble floor, gold accents, elegant furniture, crystal chandelier, high ceiling",
}

# ---------- 3.3 Quality tag & negative prompt ----------
QUALITY_TAGS = ("photorealistic, professional interior photography, architectural digest, "
                "soft natural lighting, detailed fabric and wood texture, sharp focus, high resolution")
# style vẽ minh hoạ thì không cần photorealistic -> tag riêng
NON_PHOTO_STYLES = {"Cartoon"}
NON_PHOTO_TAGS = "high quality illustration, clean vector render, detailed"

# Với phòng trống, phải nói rõ "đã bày đồ" nếu không model có xu hướng trả lại phòng trống
FURNISH_TAGS = "fully furnished, professionally staged, complete furniture set arranged in the room"

NEGATIVE_PROMPT = (
    "blurry, low quality, jpeg artifacts, distorted, deformed furniture, watermark, text, logo, "
    "unrealistic proportions, warped walls, crooked lines, extra doors, extra windows, people, "
    "duplicate objects, cluttered mess, floating furniture, oversaturated"
)
# thêm khi xử lý phòng trống
NEGATIVE_EMPTY = "empty room, unfurnished, bare floor, no furniture, vacant, plain empty walls"


def build_prompt(room_type: str, style: str, is_empty_room: bool = True) -> tuple:
    """Trả về (prompt, prompt_2).

    SDXL có 2 text encoder, mỗi cái chỉ nhận 77 token. Nhồi hết vào 1 chỗ thì
    diffusers cắt phần cuối -> mất luôn quality tag. Nên tách:
      prompt   = style + loại phòng + nội thất  (nội dung chính)
      prompt_2 = tag bày đồ + tag chất lượng    (bổ nghĩa)
    """
    assert room_type in ROOM_TYPES, f"Room type không hợp lệ. Chọn 1 trong: {list(ROOM_TYPES)}"
    assert style in STYLE_PROMPTS, f"Style không hợp lệ. Chọn 1 trong: {list(STYLE_PROMPTS)}"
    room_en, furniture = ROOM_TYPES[room_type]
    tags = NON_PHOTO_TAGS if style in NON_PHOTO_STYLES else QUALITY_TAGS

    prompt = f"{STYLE_PROMPTS[style]} {room_en} interior, {furniture}"
    prompt_2 = f"{FURNISH_TAGS}, {tags}" if is_empty_room else tags
    return prompt, prompt_2


def build_negative(is_empty_room: bool = True) -> tuple:
    """Trả về (negative_prompt, negative_prompt_2)."""
    return NEGATIVE_PROMPT, (NEGATIVE_EMPTY if is_empty_room else "")


def count_tokens(text: str):
    """Đếm token bằng chính tokenizer của pipeline. None nếu chưa load model."""
    _pipe = globals().get("pipe", None)
    if _pipe is None:
        return None
    return len(_pipe.tokenizer(text).input_ids)


def check_prompt(room_type: str, style: str, is_empty_room: bool = True, verbose: bool = True):
    """In prompt + số token, cảnh báo nếu vượt 77 (SDXL sẽ cắt phần vượt)."""
    prompt, prompt_2 = build_prompt(room_type, style, is_empty_room)
    for label, text in (("prompt", prompt), ("prompt_2", prompt_2)):
        n = count_tokens(text)
        n_str = "?" if n is None else str(n)
        flag = "  <-- VƯỢT 77 TOKEN, SDXL sẽ cắt phần cuối" if (n or 0) > 77 else ""
        if verbose:
            print(f"[{label}] {n_str} token{flag}\n  {text}\n")
    return prompt, prompt_2


print(f"{len(ROOM_TYPES)} loại phòng x {len(STYLE_PROMPTS)} style = {len(ROOM_TYPES) * len(STYLE_PROMPTS)} tổ hợp")
print("\nVí dụ (phòng trống, Phòng khách + Peaceful):\n")
check_prompt("Phòng khách", "Peaceful", is_empty_room=True)


## 4. Hàm xử lý chính

Hai chế độ, khác nhau ở chỗ dùng ControlNet nào:

| Ảnh gốc | ControlNet | Vì sao |
|---|---|---|
| **Phòng trống** | Canny (giữ đường kiến trúc) | Depth map của phòng trống toàn mặt phẳng -> nếu bám theo depth thì model không dám tạo khối 3D, đồ đạc ra bẹt như dán lên tường. Canny chỉ giữ khung cửa sổ / góc tường / chân tường, chừa toàn bộ thể tích trống cho model bày đồ. |
| **Phòng đã có đồ** | Canny + Depth | Lúc này depth map có thông tin khối thật của đồ đạc, dùng để giữ bố cục cũ khi đổi style. |

Ảnh được resize **giữ đúng tỉ lệ gốc** về ~1 megapixel (kích thước SDXL được train), không bóp về vuông.


In [ ]:
import cv2

def fit_size(w: int, h: int, target_px: int = 1024 * 1024) -> tuple:
    """Giữ tỉ lệ gốc, scale về ~target_px, làm tròn về bội số 8 (yêu cầu của SDXL)."""
    ar = w / h
    new_h = (target_px / ar) ** 0.5
    new_w = ar * new_h
    return (max(512, int(round(new_w / 8) * 8)), max(512, int(round(new_h / 8) * 8)))


def make_canny(image: Image.Image, low: int = 80, high: int = 180) -> Image.Image:
    """Đường biên kiến trúc. Ngưỡng cao -> ít nhiễu, chỉ còn cạnh tường/cửa sổ rõ."""
    arr = cv2.Canny(np.array(image), low, high)
    return Image.fromarray(np.stack([arr] * 3, axis=-1))


def make_depth(image: Image.Image) -> Image.Image:
    assert depth_estimator is not None, "Chưa load depth estimator. Bật load_depth_controlnet ở bước 2 và chạy lại."
    return depth_estimator(image).resize(image.size)


def redesign_room(
    input_image_path: str,
    room_type: str,
    style: str,
    is_empty_room: bool = True,
    canny_scale: float = 0.55,      # 0.3-0.8. Cao = bám sát kiến trúc, thấp = model tự do hơn
    depth_scale: float = 0.55,      # chỉ dùng khi is_empty_room = False
    guidance_scale: float = 6.0,    # RealVisXL/Juggernaut thích 4-7, đừng đẩy lên 12
    steps: int = 30,
    seed: int = 42,
    target_px: int = 1024 * 1024,
):
    src = Image.open(input_image_path).convert("RGB")
    size = fit_size(*src.size, target_px=target_px)
    room_image = src.resize(size, Image.LANCZOS)

    # Chọn control image theo đúng thứ tự controlnet đã nạp ở bước 2
    canny_image = make_canny(room_image)
    if is_empty_room:
        # tắt depth bằng cách cho weight = 0 (vẫn phải truyền đủ ảnh cho mọi controlnet đã nạp)
        control_images = [canny_image]
        control_scales = [canny_scale]
        depth_image = None
        if "depth" in CONTROL_ORDER:
            depth_image = Image.new("RGB", size, (0, 0, 0))
            control_images.append(depth_image)
            control_scales.append(0.0)
    else:
        assert "depth" in CONTROL_ORDER, "Cần bật load_depth_controlnet ở bước 2 để xử lý phòng đã có đồ."
        depth_image = make_depth(room_image)
        control_images = [canny_image, depth_image]
        control_scales = [canny_scale, depth_scale]

    generator = torch.Generator(device="cpu").manual_seed(seed)

    prompt, prompt_2 = build_prompt(room_type, style, is_empty_room)
    negative, negative_2 = build_negative(is_empty_room)

    result = pipe(
        prompt=prompt,
        prompt_2=prompt_2,
        negative_prompt=negative,
        negative_prompt_2=negative_2,
        image=control_images,
        controlnet_conditioning_scale=control_scales,
        num_inference_steps=steps,
        guidance_scale=guidance_scale,
        width=size[0],
        height=size[1],
        generator=generator,
    ).images[0]

    control_preview = depth_image if (depth_image is not None and not is_empty_room) else canny_image
    return room_image, control_preview, result


## 5. Upload ảnh phòng & render

Bấm **Choose Files** để chọn ảnh phòng từ máy (jpg/png/webp). Chọn nhiều ảnh được, notebook xử lý lần lượt.

Giải thích từng thông số nằm ngay trong form bên dưới. Bảng tra nhanh khi kết quả chưa đạt:

| Triệu chứng | Sửa |
|---|---|
| Phòng vẫn trống / rất ít đồ | hạ `canny_scale` về **0.30-0.45** |
| Tường, cửa sổ bị méo / lệch | nâng `canny_scale` lên **0.70-0.85** |
| Ảnh trông "AI", màu bệt | hạ `guidance_scale` về **4.5-5.5**, tăng `steps` lên **40** |
| Đồ đạc đè lên cửa sổ | nâng `canny_scale`, và mô tả rõ hơn ở `ROOM_TYPES` (bước 3) |
| Đổi style nhưng bố cục đồ cũ vẫn còn | chỉ xảy ra ở chế độ *Phòng đã có đồ* — hạ `depth_scale` về 0.3 |
| `CUDA out of memory` | đổi `resolution` sang **0.6 MP**, hoặc tắt `load_depth_controlnet` ở bước 2 |
| Ảnh nào cũng giống nhau | đổi `seed` (mỗi seed = 1 phương án bố trí khác) |


In [ ]:
#@title Upload ảnh phòng và render { display-mode: "form" }

#@markdown ### 1. Phòng gì, theo phong cách nào
#@markdown `room_type` quyết định **model đưa đồ gì vào phòng** (giường / sofa / bàn ăn...) — xem danh sách đồ của từng loại trong `ROOM_TYPES` ở bước 3.
room_type = "Phòng khách"  #@param ["Phòng khách", "Phòng ngủ", "Phòng ăn", "Phòng tắm", "Bếp", "Phòng chơi game", "Nhà hàng", "Văn phòng tại gia", "Quán cà phê", "Văn phòng"]
#@markdown `chosen_style` quyết định **màu sắc, vật liệu, ánh sáng** của đồ đạc đó. Không đổi loại đồ, chỉ đổi "gu".
chosen_style = "Peaceful"  #@param ["Peaceful", "Farmhouse", "Clean Bright", "Contemporary", "Fresh Airy", "Eclectic", "Elegant", "Minimalist", "Minimal Tranquil", "Cartoon", "Scandinavian", "Simple Calm", "Bright Soothing", "Cyberpunk", "Rustic", "Compact Calm", "Classic Graceful", "Traditional", "Industrial", "Mid-Century", "Japandi", "Luxury Classic"]

#@markdown ---
#@markdown ### 2. Ảnh bạn upload là phòng trống hay đã có đồ?
#@markdown Đây là lựa chọn **quan trọng nhất**, nó đổi hẳn cách giữ kiến trúc:
#@markdown - **Phòng trống** → chỉ dùng ControlNet **Canny**, tắt Depth. Phải chọn cái này cho ảnh phòng trắng trơn: depth map của phòng trống toàn mặt phẳng, nếu bám theo nó thì model không dám tạo khối 3D và đồ đạc ra bẹt như dán lên tường.
#@markdown - **Phòng đã có đồ** → **Canny + Depth**. Lúc này depth map có khối thật của đồ cũ, dùng để giữ bố cục khi đổi style.
anh_goc = "Phòng trống"  #@param ["Phòng trống", "Phòng đã có đồ"]

#@markdown ---
#@markdown ### 3. Độ phân giải
#@markdown Ảnh luôn được **giữ đúng tỉ lệ gốc**, chỉ scale về mức pixel bên dưới (SDXL được train quanh 1 MP nên ra ảnh 1 MP là đẹp nhất).
#@markdown - `1.0 MP` — ảnh 16:9 ra ~1368x768. Chậm hơn, nét hơn.
#@markdown - `0.6 MP` — ảnh 16:9 ra ~1056x592. Nhanh hơn ~40%, dùng khi bị hết VRAM hoặc đang thử style.
resolution = "1.0 MP (chất lượng)"  #@param ["1.0 MP (chất lượng)", "0.6 MP (nhanh)"]

#@markdown ---
#@markdown ### 4. Hai thanh ControlNet — mức "bám" vào ảnh gốc
#@markdown **`canny_scale`** = model phải tôn trọng **đường nét kiến trúc** của ảnh gốc mạnh đến đâu. Canny map là ảnh đường viền trắng/đen: khung cửa sổ, góc tường, chân tường, đường trần.
#@markdown - **cao (0.8-1.0)** — tường/cửa sổ đứng đúng chỗ tuyệt đối, nhưng model bị bó, thêm được ít đồ và đồ hay bị bẹt.
#@markdown - **0.5-0.6 (mặc định)** — cân bằng: kiến trúc giữ được, vẫn đủ tự do bày đồ có khối.
#@markdown - **thấp (0.2-0.4)** — bày đồ rất thoáng, nhiều đồ, đổi cả màu tường; đổi lại cửa sổ có thể lệch hoặc biến dạng.
canny_scale = 0.55  #@param {type:"slider", min:0.2, max:1.0, step:0.05}
#@markdown **`depth_scale`** = mức bám vào **khối 3D / bố cục đồ đạc** của ảnh gốc. **Chỉ có tác dụng khi chọn "Phòng đã có đồ"** — ở chế độ phòng trống nó bị đặt về 0 tự động.
#@markdown - cao (0.7-1.0) — giữ gần như nguyên vị trí và kích thước đồ cũ, chỉ đổi vật liệu/màu.
#@markdown - thấp (0.2-0.4) — cho phép thay đồ khác kiểu, khác kích thước.
depth_scale = 0.55  #@param {type:"slider", min:0.0, max:1.0, step:0.05}

#@markdown ---
#@markdown ### 5. Thông số sinh ảnh
#@markdown **`guidance_scale` (CFG)** = model phải tuân prompt sát đến đâu. Checkpoint kiểu RealVisXL/Juggernaut thích khoảng **4-7**; đẩy lên 10-12 là ảnh cháy màu, viền gắt, trông "AI" ngay.
guidance_scale = 6.0  #@param {type:"slider", min:3.0, max:10.0, step:0.5}
#@markdown **`steps`** = số bước khử nhiễu. 25-30 là đủ; 40+ nét hơn chút nhưng lâu gần gấp rưỡi; dưới 20 là đồ đạc bắt đầu nhoè, thiếu chi tiết.
steps = 30  #@param {type:"integer"}
#@markdown **`seed`** = số khởi tạo nhiễu. **Cùng seed + cùng thông số = ra đúng ảnh cũ**, nên giữ nguyên seed khi đang tinh chỉnh các thanh trượt để so sánh cho công bằng. Đổi seed khi muốn xem phương án bố trí đồ khác.
seed = 42  #@param {type:"integer"}

import os, time
from google.colab import files

is_empty_room = (anh_goc == "Phòng trống")
target_px = 1024 * 1024 if resolution.startswith("1.0") else int(0.6 * 1024 * 1024)

print("Chọn 1 hoặc nhiều ảnh phòng để upload...")
uploaded = files.upload()

VALID_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
image_paths = []
for name in uploaded.keys():
    if not name.lower().endswith(VALID_EXT):
        print(f"  Bỏ qua (không phải ảnh): {name}")
        continue
    path = os.path.join("/content", name)
    with open(path, "wb") as f:
        f.write(uploaded[name])
    image_paths.append(path)

assert image_paths, "Chưa có ảnh nào được upload. Chạy lại cell và chọn file ảnh."

print(f"\nChế độ: {anh_goc} | ControlNet: {'canny' if is_empty_room else 'canny + depth'}"
      f" | canny={canny_scale}" + ("" if is_empty_room else f" depth={depth_scale}") + "\n")
check_prompt(room_type, chosen_style, is_empty_room)

results = []   # [(tên file, original, control, output), ...]
for path in image_paths:
    t0 = time.time()
    print(f"Đang xử lý: {os.path.basename(path)} | {room_type} | {chosen_style} ...")
    original, control, output = redesign_room(
        path,
        room_type,
        chosen_style,
        is_empty_room=is_empty_room,
        canny_scale=canny_scale,
        depth_scale=depth_scale,
        guidance_scale=guidance_scale,
        steps=steps,
        seed=seed,
        target_px=target_px,
    )
    results.append((os.path.basename(path), original, control, output))
    print(f"  xong sau {time.time() - t0:.0f}s | {output.size[0]}x{output.size[1]}")

# giữ tương thích với cell hiển thị bên dưới (dùng ảnh cuối cùng)
input_path = image_paths[-1]
_, original, control, output = results[-1]
print(f"\nXong {len(results)} ảnh.")


## 6. Hiển thị kết quả: gốc / control map / kết quả


In [ ]:
from PIL import Image as PILImage
from IPython.display import display

def show_side_by_side(*images, row_height=384):
    """Ghép ngang, chuẩn hoá theo chiều cao để không méo ảnh."""
    imgs = []
    for im in images:
        im = im.convert("RGB")
        w = int(im.width * row_height / im.height)
        imgs.append(im.resize((w, row_height), PILImage.LANCZOS))
    total_width = sum(im.width for im in imgs)
    combined = PILImage.new("RGB", (total_width, row_height), (255, 255, 255))
    x = 0
    for im in imgs:
        combined.paste(im, (x, 0))
        x += im.width
    return combined

for name, orig, ctrl, out in results:
    stem = name.rsplit(".", 1)[0]
    out.save(f"/content/{stem}_result.jpg", quality=95)
    combined = show_side_by_side(orig, ctrl, out)
    combined.save(f"/content/{stem}_compare.jpg", quality=95)
    print(f"{name} -> /content/{stem}_result.jpg  (+ _compare.jpg)")
    display(combined)
    display(out)
